In [ ]:
from pathlib import Path
from datasets import Dataset
from transformers import (
    BertConfig,
    BertForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import math
import numpy as np

In [ ]:
DATA_FILE = "./data.txt"
MODEL_DIR = "./model"

SEP = "<|eos|>"
MIN_DOC_CHARS = 200
MAX_LENGTH = 256
TEST_SIZE = 0.01
SEED = 42

In [ ]:
def iter_docs(path, sep=SEP, min_doc_chars=MIN_DOC_CHARS):
    acc = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if line == sep:
                doc = "\n".join(acc).strip()
                acc = []

                if doc and len(doc) >= min_doc_chars:
                    yield {"text": doc}
            else:
                acc.append(line)

        if acc:
            doc = "\n".join(acc).strip()
            if doc and len(doc) >= min_doc_chars:
                yield {"text": doc}

In [ ]:
dataset = Dataset.from_generator(
    iter_docs,
    gen_kwargs={"path": DATA_FILE}
)

dataset = dataset.shuffle(seed=SEED)
raw_datasets = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)

print(raw_datasets)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print(tokenizer.vocab_size)
print(tokenizer.special_tokens_map)

In [ ]:
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_special_tokens_mask=True,
    )

tokenized_datasets = raw_datasets.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing dataset"
)

print(tokenized_datasets)

In [ ]:
config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=192,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=768,
    hidden_act="gelu",
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    max_position_embeddings=MAX_LENGTH,
    type_vocab_size=2,
    pad_token_id=tokenizer.pad_token_id,
    layer_norm_eps=1e-12,
)

model = BertForMaskedLM(config)

model_size = sum(t.numel() for t in model.parameters())
print(f"Transformer size: {model_size/1000**2:.1f}M parameters")

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    overwrite_output_dir=True,

    do_train=True,
    do_eval=True,

    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2, # 4 for mid

    num_train_epochs=5,

    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.06,

    lr_scheduler_type="cosine",

    fp16=True,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    eval_steps=10000,
    save_steps=10000,
    logging_steps=10000,

    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(metrics)

if "eval_loss" in metrics:
    print("perplexity:", round(math.exp(metrics["eval_loss"]), 2))

In [ ]:
trainer.save_model(MODEL_DIR)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
import matplotlib.pyplot as plt
logs = trainer.state.log_history

train_steps = []
train_losses = []

eval_steps = []
eval_losses = []

for log in logs:
    if 'loss' in log and 'eval_loss' not in log:
        train_steps.append(log['step'])
        train_losses.append(log['loss'])

    if 'eval_loss' in log:
        eval_steps.append(log['step'])
        eval_losses.append(log['eval_loss'])


plt.figure(figsize=(12, 6))

plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Validation Loss", linewidth=2)

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)

plt.show()